In [3]:

from datetime import datetime, timedelta
from api.country import get_country_from_city, get_country_info
from api.places import get_lat_lon, get_places_nearby  # we will wrap your places.py later
from api.weather import get_weather_forecast  # wrap weather.py later
from api.tips import get_travel_tips

city = "paris"
duration = 3
start_date = "2025-08-25"
 # --- Resolve country info ---
country_name = get_country_from_city(city)

country_info = get_country_info(country_name)

# --- Resolve lat/lon ---
lat, lon = get_lat_lon(city)


# --- Determine start and end dates ---
if start_date:
    start = datetime.strptime(start_date, "%Y-%m-%d")
else:
    start = datetime.today()
end = start + timedelta(days=duration-1)

# --- Get weather forecast ---
weather_data = get_weather_forecast(lat, lon, 3,start.strftime("%Y-%m-%d"))

# --- Get nearby attractions / restaurants ---
places = get_places_nearby(lat, lon)
travel_tips = get_travel_tips(city)

print(places)
print(weather_data)
print(country_info)
print(travel_tips)

[{'name': "Fondation Cartier pour l'art contemporain", 'categories': ['entertainment', 'entertainment.museum', 'fee', 'wheelchair', 'wheelchair.yes'], 'address': "Fondation Cartier pour l'art contemporain"}, {'name': 'Sainte-Chapelle', 'categories': ['building', 'building.historic', 'building.place_of_worship', 'building.tourism', 'entertainment', 'entertainment.museum', 'fee', 'heritage', 'religion', 'religion.place_of_worship', 'religion.place_of_worship.christianity', 'tourism.sights.place_of_worship', 'tourism.sights.place_of_worship.church', 'wheelchair', 'wheelchair.yes'], 'address': 'Sainte-Chapelle'}, {'name': 'Musée des beaux-arts de la Ville de Paris', 'categories': ['building', 'building.historic', 'building.tourism', 'entertainment', 'entertainment.museum', 'heritage', 'internet_access', 'internet_access.free', 'no_fee', 'no_fee.no', 'wheelchair', 'wheelchair.yes'], 'address': 'Musée des beaux-arts de la Ville de Paris'}, {'name': 'Musée Carnavalet', 'categories': ['access'

In [5]:
def flatten_weather(data):
    daily = data["daily"]
    days = []

    for date, t_max, t_min, h_max, h_min, code in zip(
        daily["time"],
        daily["temperature_2m_max"],
        daily["temperature_2m_min"],
        daily["relative_humidity_2m_max"],
        daily["relative_humidity_2m_min"],
        daily["weathercode"],
    ):
        day = {
            "date": date,
            "temperature_max": float(t_max),
            "temperature_min": float(t_min),
            "humidity_max": int(h_max),
            "humidity_min": int(h_min),
            "weather_code": int(code),
        }
        days.append(day)

    return days


flattened = flatten_weather(weather_data)
print(flattened)


[{'date': '2025-08-25', 'temperature_max': 28.1, 'temperature_min': 15.2, 'humidity_max': 64, 'humidity_min': 26, 'weather_code': 3}, {'date': '2025-08-26', 'temperature_max': 29.4, 'temperature_min': 17.9, 'humidity_max': 62, 'humidity_min': 28, 'weather_code': 3}, {'date': '2025-08-27', 'temperature_max': 26.6, 'temperature_min': 19.4, 'humidity_max': 73, 'humidity_min': 31, 'weather_code': 3}]


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain_ollama import OllamaLLM
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler


# 1. Forecast Teller Chain
def forecast_chain(llm):
    forecast_prompt = PromptTemplate(
        input_variables=["city", "weather_data"],
        template="""
You are a forecast teller.

Weather forecast (JSON): {weather_data}

Your task:
- For each day in weather_data, extract values EXACTLY as given (temperature_max, temperature_min, humidity_max, humidity_min, weather_code).
- Do not round, change, or assume values.
- Translate weather_code into human readable form using the table below:
Weather Code Reference Table:
0 = Clear sky/Sunny | Clothing: Light & breathable | Best for outdoor activities  
1, 2, 3 = Mainly clear / Partly cloudy / Overcast | Clothing: Light jacket | Outdoor possible, but watch skies  
45, 48 = Fog / Depositing rime fog | Clothing: Warm, layers | Travel visibility reduced, caution for morning plans  
51, 53, 55 = Light / Moderate / Dense drizzle | Clothing: Waterproof jacket | Outdoor activities may be less enjoyable  
56, 57 = Freezing drizzle | Clothing: Very warm + waterproof | Travel may be hazardous  
61, 63, 65 = Slight / Moderate / Heavy rain | Clothing: Raincoat & umbrella | Outdoor activities affected  
66, 67 = Freezing rain | Clothing: Heavy waterproof | Dangerous conditions possible  
71, 73, 75 = Slight / Moderate / Heavy snowfall | Clothing: Winter gear | Outdoor may be difficult  
77 = Snow grains | Clothing: Heavy coat | Travel impact  
80, 81, 82 = Rain showers (slight, moderate, violent) | Clothing: Rain gear | Unpredictable, plan indoor alternatives  
85, 86 = Snow showers (slight, heavy) | Clothing: Winter clothing | Risk of disruption  
95 = Thunderstorm (slight/moderate) | Clothing: Light + rain gear | Outdoor unsafe  
96, 99 = Thunderstorm with hail (slight/heavy) | Clothing: Protective gear | Avoid outdoor plans  
Output must be strict JSON in this format:
{{
  "forecast": [
    {{
      "date": "...",
      "temperature_max": "...",
      "temperature_min": "...",
      "humidity_max": "...",
      "humidity_min": "...",
      "weather_description": "...",
      
      
    }}
  ]
}}
"""
    )
    return LLMChain(llm=llm, prompt=forecast_prompt, output_key="forecast_json")


# 2. Travel Planner Chain
def planner_chain(llm):
    planner_prompt = PromptTemplate(
        input_variables=["city", "country_info", "forecast_json", "places", "travel_tips", "duration"],
        template="""
You are a travel planner. Create a {duration}-day itinerary for {city}.

Country details: {country_info}
Weather forecast (verified): {forecast_json}
Nearby attractions/restaurants: {places}
Travel tips: {travel_tips}

Format clearly with:
- Morning, Afternoon, Evening activities
- At the start of each day, include the weather note (use values from forecast_json, do not change them).
- Local tips at the end of each day.
- Use the weather_description, suggested_clothing, and travel_impact from forecast_json.
- Tell temperature and humidity exactly for each respective date, verifying from forecast_json without rounding or modifying.
Do not invent temperatures, humidity, or weather descriptions. ONLY use forecast_json values.
"""
    )
    return LLMChain(llm=llm, prompt=planner_prompt, output_key="itinerary")


# Main function to run both chains
def generate_itineraryss(city, country_info, weather_data, places, travel_tips, duration=3):
    llm = OllamaLLM(
        model="mistral",
        streaming=True,
        callbacks=[StreamingStdOutCallbackHandler()]
    )

    # Build chains
    forecast = forecast_chain(llm)
    planner = planner_chain(llm)

    # Sequential chain
    overall_chain = SequentialChain(
        chains=[forecast, planner],
        input_variables=["city", "country_info", "weather_data", "places", "travel_tips", "duration"],
        output_variables=["forecast_json", "itinerary"],
        verbose=True
    )

    return overall_chain(
        {
            "city": city,
            "country_info": country_info,
            "weather_data": weather_data,
            "places": places,
            "travel_tips": travel_tips,
            "duration": duration,
        }
    )




In [7]:
generate_itineraryss(city,country_info,flattened, places, travel_tips, duration=3)

C:\Users\abdur\AppData\Local\Temp\ipykernel_53208\1427695728.py:51: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  return LLMChain(llm=llm, prompt=forecast_prompt, output_key="forecast_json")
C:\Users\abdur\AppData\Local\Temp\ipykernel_53208\1427695728.py:98: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return overall_chain(




> Entering new SequentialChain chain...
 Here is the JSON output for your weather forecast:

```json
{
  "forecast": [
    {
      "date": "2025-08-25",
      "temperature_max": "28.1",
      "temperature_min": "15.2",
      "humidity_max": "64",
      "humidity_min": "26",
      "weather_description": "Clear sky/Sunny"
    },
    {
      "date": "2025-08-26",
      "temperature_max": "29.4",
      "temperature_min": "17.9",
      "humidity_max": "62",
      "humidity_min": "28",
      "weather_description": "Mainly clear / Partly cloudy / Overcast"
    },
    {
      "date": "2025-08-27",
      "temperature_max": "26.6",
      "temperature_min": "19.4",
      "humidity_max": "73",
      "humidity_min": "31",
      "weather_description": "Mainly clear / Partly cloudy / Overcast"
    }
  ]
}
``` Day 1 - Monday (July 26)
Weather Note: Temperature: 24°C, Humidity: 50%, Weather Description: Partly Cloudy

Morning Activities:
- Start your day with a visit to the iconic Eiffel Tower. The e

{'city': 'paris',
 'country_info': {'name': 'France',
  'official_name': 'French Republic',
  'capital': 'Paris',
  'region': 'Europe',
  'subregion': 'Western Europe',
  'languages': ['French'],
  'currencies': ['EUR'],
  'timezones': ['UTC-10:00',
   'UTC-09:30',
   'UTC-09:00',
   'UTC-08:00',
   'UTC-04:00',
   'UTC-03:00',
   'UTC+01:00',
   'UTC+02:00',
   'UTC+03:00',
   'UTC+04:00',
   'UTC+05:00',
   'UTC+10:00',
   'UTC+11:00',
   'UTC+12:00'],
  'population': 67391582,
  'flag': 'https://flagcdn.com/w320/fr.png'},
 'weather_data': [{'date': '2025-08-25',
   'temperature_max': 28.1,
   'temperature_min': 15.2,
   'humidity_max': 64,
   'humidity_min': 26,
   'weather_code': 3},
  {'date': '2025-08-26',
   'temperature_max': 29.4,
   'temperature_min': 17.9,
   'humidity_max': 62,
   'humidity_min': 28,
   'weather_code': 3},
  {'date': '2025-08-27',
   'temperature_max': 26.6,
   'temperature_min': 19.4,
   'humidity_max': 73,
   'humidity_min': 31,
   'weather_code': 3}],
 '

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_ollama import OllamaLLM
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
import json

# --- Deterministic weather mapping (no model involved) ---
WEATHER_MAP = {
    0: ("Clear sky/Sunny", "Light & breathable"),
    1: ("Mainly clear", "Light jacket"),
    2: ("Partly cloudy", "Light jacket"),
    3: ("Overcast", "Light jacket"),
    45: ("Fog", "Warm layers"),
    48: ("Fog", "Warm layers"),
    51: ("Light drizzle", "Waterproof jacket"),
    53: ("Moderate drizzle", "Waterproof jacket"),
    55: ("Dense drizzle", "Waterproof jacket"),
    61: ("Slight rain", "Raincoat & umbrella"),
    63: ("Moderate rain", "Raincoat & umbrella"),
    65: ("Heavy rain", "Raincoat & umbrella"),
    71: ("Slight snowfall", "Winter gear"),
    73: ("Moderate snowfall", "Winter gear"),
    75: ("Heavy snowfall", "Winter gear"),
    77: ("Snow grains", "Heavy coat"),
    80: ("Slight rain showers", "Rain gear"),
    81: ("Moderate rain showers", "Rain gear"),
    82: ("Violent rain showers", "Rain gear"),
    85: ("Slight snow showers", "Winter clothing"),
    86: ("Heavy snow showers", "Winter clothing"),
    95: ("Thunderstorm", "Rain gear"),
    96: ("Thunderstorm with hail (slight)", "Protective gear"),
    99: ("Thunderstorm with hail (heavy)", "Protective gear"),
}

def build_verified_weather(weather_data):
    """Return list of per-day dicts with exact numbers + derived description/clothing."""
    verified = []
    for d in weather_data:
        code = int(d["weather_code"])
        desc, clothing = WEATHER_MAP.get(code, ("See weather_code", "Weather-appropriate"))
        verified.append({
            "date": d["date"],
            "weather_code": code,
            "weather_description": desc,
            "suggested_clothing": clothing,
            "temperature_max": d["temperature_max"],
            "temperature_min": d["temperature_min"],
            "humidity_max": d["humidity_max"],
            "humidity_min": d["humidity_min"],
        })
    return verified

# --- Single-day planner prompt (forces the model to COPY your weather line) ---
DAY_PROMPT = PromptTemplate(
    input_variables=["city", "country_info", "places", "travel_tips", "day_json"],
    template="""
You are a travel planner for {city}. Use the provided JSON.

STRICT WEATHER NOTE (copy EXACTLY as provided; do not change any number or text):
From day_json -> build this line and copy it verbatim:
Weather note: {{"date": "<date>", "description": "<weather_description>", "code": <weather_code>, "temperature_max": <temperature_max>, "temperature_min": <temperature_min>, "humidity_max": <humidity_max>, "humidity_min": <humidity_min>, "clothing": "<suggested_clothing>"}}

day_json: {day_json}

Context:
- Country details: {country_info}
- Nearby attractions/restaurants: {places}
- Travel tips: {travel_tips}

Output:
1) First line must be the exact WEATHER NOTE as specified (same keys, same order, same values).
2) Then provide "Morning", "Afternoon", "Evening" activities tailored to this date's weather.
3) End with one local tip.

Rules:
- Do not invent or modify temperatures/humidity/code/description.
- Plan only for the given date in day_json; no other days.
"""
)

def make_day_chain(llm):
    return LLMChain(llm=llm, prompt=DAY_PROMPT, output_key="day_plan")

# --- Main: plan per-day and stitch results so nothing is skipped ---
def generate_itineraryss(city, country_info, weather_data, places, travel_tips, duration=3):
    llm = OllamaLLM(model="mistral", streaming=True, callbacks=[StreamingStdOutCallbackHandler()])

    verified_weather = build_verified_weather(weather_data)
    # optionally trim to requested duration to avoid mismatch
    verified_weather = verified_weather[:duration]

    day_chain = make_day_chain(llm)
    day_outputs = []

    for d in verified_weather:
        day_json = json.dumps(d, ensure_ascii=False)
        res = day_chain({
            "city": city,
            "country_info": country_info,
            "places": places,
            "travel_tips": travel_tips,
            "day_json": day_json,
        })
        # Ensure the exact weather line made it through (fallback to deterministic line if not)
        exact_line = (
            f'Weather note: {{"date": "{d["date"]}", "description": "{d["weather_description"]}", '
            f'"code": {d["weather_code"]}, "temperature_max": {d["temperature_max"]}, '
            f'"temperature_min": {d["temperature_min"]}, "humidity_max": {d["humidity_max"]}, '
            f'"humidity_min": {d["humidity_min"]}, "clothing": "{d["suggested_clothing"]}"}}'
        )
        text = res["day_plan"]
        if "Weather note:" not in text or d["date"] not in text:
            # harden against any slip: prepend the exact note
            text = exact_line + "\n" + text
        day_outputs.append(f"Day for {d['date']}\n{text}")

    return "\n\n".join(day_outputs)


In [11]:
generate_itinerary(city,country_info,flattened, places, travel_tips, duration=3)

 Day 1: Weather Note - Mainly clear/Partly cloudy (weather_code: 2)

Morning Activities:
- Start your day at the Louvre Museum to explore some of the world's most famous artworks such as the Mona Lisa and the Winged Victory of Samothrace. Wear light clothing, but remember a light jacket for potential chillier indoor spaces.

Afternoon Activities:
- Head over to the Seine River and hop on a Bateaux Mouches cruise to get a unique perspective of iconic Parisian landmarks like the Eiffel Tower, Notre Dame Cathedral, and the Orsay Museum. Dress for variable weather as you'll be outside most of the time.

Evening Activities:
- Spend an evening at the Eiffel Tower. You can choose to dine at one of its restaurants or simply admire the city lights from below. Keep a light jacket handy, as temperatures may drop in the evening.

Day 2: Weather Note - Rain (weather_code: 61)

Morning Activities:
- Visit the Palace of Versailles and its stunning gardens. Make sure to wear a raincoat and carry an um

{'city': 'paris',
 'country_info': {'name': 'France',
  'official_name': 'French Republic',
  'capital': 'Paris',
  'region': 'Europe',
  'subregion': 'Western Europe',
  'languages': ['French'],
  'currencies': ['EUR'],
  'timezones': ['UTC-10:00',
   'UTC-09:30',
   'UTC-09:00',
   'UTC-08:00',
   'UTC-04:00',
   'UTC-03:00',
   'UTC+01:00',
   'UTC+02:00',
   'UTC+03:00',
   'UTC+04:00',
   'UTC+05:00',
   'UTC+10:00',
   'UTC+11:00',
   'UTC+12:00'],
  'population': 67391582,
  'flag': 'https://flagcdn.com/w320/fr.png'},
 'weather_data': '[\n  {\n    "date": "2025-08-25",\n    "temperature_max": 28.1,\n    "temperature_min": 15.2,\n    "humidity_max": 64,\n    "humidity_min": 26,\n    "weather_code": 3\n  },\n  {\n    "date": "2025-08-26",\n    "temperature_max": 29.4,\n    "temperature_min": 17.9,\n    "humidity_max": 62,\n    "humidity_min": 28,\n    "weather_code": 3\n  },\n  {\n    "date": "2025-08-27",\n    "temperature_max": 26.6,\n    "temperature_min": 19.4,\n    "humidity_